# Datengenerierung und Zugriff
Hier möchte ich Funktionen für verschiedene Parameter kombinieren. Außerdem möchte ich die Generierung der Testdaten von der Abfrage trennen damit die Generierung später einfach durch eine Datenbankabfrage ersetzt werden kann und die Abfrage kürzer und übersichtlicher wird.

In [3]:
import pandas as pd
import numpy as np
from typing import Literal

## Generate

In [4]:
Direction = Literal["consumption", "generation"]
Frequency = Literal["15min", "1h"]

TZ = "Europe/Vienna"

def create_data(
    meter_id: str,
    direction: Direction,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    freq: Frequency,
    forecast: bool
) -> pd.DataFrame:
    timestamps = pd.date_range(start=start_date, end=end_date, freq=freq, inclusive="left")
    
    base = np.sin(np.linspace(0, 2*np.pi, len(timestamps))) + 1

    rng = np.random.default_rng(42)
    noise = rng.normal(0, 0.1, len(timestamps))
    
    values = (base + noise).clip(min=0)
    
    df = pd.DataFrame({
        "timestamp": timestamps,
        "meter_id": meter_id,
        "forecast": forecast,
        "direction": direction,
        "value": values
    })

    return df.astype({
        "meter_id": "string",
        "forecast": "bool",
        "direction": "string",
        "value": "float64"
    })

def create_actual_and_forecast(
    meter_id: str,
    direction: Direction,
    start_date: pd.Timestamp
) -> pd.DataFrame:
    now = pd.Timestamp.now(tz=TZ)
    today_00 = now.normalize()
    yesterday_00 = today_00 - pd.Timedelta(days=1)
    day_after_tomorrow_00 = today_00 + pd.Timedelta(days=2)

    df_actual = create_data(
        meter_id=meter_id,
        direction=direction,
        start_date=start_date,
        end_date=yesterday_00,
        freq="15min",
        forecast=False
    )

    df_forecast = create_data(
        meter_id=meter_id,
        direction=direction,
        start_date=yesterday_00,
        end_date=day_after_tomorrow_00,
        freq="1h",
        forecast=True
    )

    return pd.concat([df_actual, df_forecast])

def create_testdata() -> pd.DataFrame:
    # start_date = pd.Timestamp("2026-04-10", tz=TZ).normalize()
    start_date = pd.Timestamp.now(tz=TZ) - pd.Timedelta(days=3)

    mp_1 = create_actual_and_forecast("mp_1", "consumption", start_date)
    mp_2 = create_actual_and_forecast("mp_2", "generation", start_date)

    return pd.concat([mp_1, mp_2])

df = create_testdata()

print(df.to_string())

                           timestamp meter_id  forecast    direction     value
0   2026-04-10 21:31:54.501061+02:00     mp_1     False  consumption  1.030472
1   2026-04-10 21:46:54.501061+02:00     mp_1     False  consumption  0.955806
2   2026-04-10 22:01:54.501061+02:00     mp_1     False  consumption  1.194439
3   2026-04-10 22:16:54.501061+02:00     mp_1     False  consumption  1.272613
4   2026-04-10 22:31:54.501061+02:00     mp_1     False  consumption  1.041977
5   2026-04-10 22:46:54.501061+02:00     mp_1     False  consumption  1.164537
6   2026-04-10 23:01:54.501061+02:00     mp_1     False  consumption  1.364159
7   2026-04-10 23:16:54.501061+02:00     mp_1     False  consumption  1.375112
8   2026-04-10 23:31:54.501061+02:00     mp_1     False  consumption  1.458962
9   2026-04-10 23:46:54.501061+02:00     mp_1     False  consumption  1.427595
10  2026-04-11 00:01:54.501061+02:00     mp_1     False  consumption  1.651260
11  2026-04-11 00:16:54.501061+02:00     mp_1     Fa

## Access

In [ ]:


def get_data_by_meter_id(df, meter_id=None, direction=None, forecast=None):
    result = df.copy()

    if meter_id is not None:
        result = result[result["meter_id"] == meter_id]

    if direction is not None:
        result = result[result["direction"] == direction]
    
    if forecast is not None:
        result = result[result["forecast"] == forecast]
    
    return result

def total_energy(df):
    return df["value"].sum()

In [24]:
get_data_by_meter_id(df, direction="consumption", forecast=True)

,timestamp,meter_id,forecast,direction,value
0,2026-04-12 00:00:00+02:00,mp_1,True,consumption,1.030472
1,2026-04-12 01:00:00+02:00,mp_1,True,consumption,0.984382
2,2026-04-12 02:00:00+02:00,mp_1,True,consumption,1.251114
3,2026-04-12 03:00:00+02:00,mp_1,True,consumption,1.356435
4,2026-04-12 04:00:00+02:00,mp_1,True,consumption,1.151532
...,...,...,...,...,...
67,2026-04-14 19:00:00+02:00,mp_1,True,consumption,0.607129
68,2026-04-14 20:00:00+02:00,mp_1,True,consumption,0.823419
69,2026-04-14 21:00:00+02:00,mp_1,True,consumption,0.804801
70,2026-04-14 22:00:00+02:00,mp_1,True,consumption,0.784051


In [27]:
def to_wide(df):
    return df.pivot_table(
        index=["timestamp", "meter_id", "forecast"],
        columns="direction",
        values="value"
    ).reset_index()

In [28]:
df_test = to_wide(df)

df_test

direction,timestamp,meter_id,forecast,consumption,generation
0,2026-04-10 21:31:54.501061+02:00,mp_1,False,1.030472,NaN
1,2026-04-10 21:31:54.501061+02:00,mp_2,False,NaN,1.030472
2,2026-04-10 21:46:54.501061+02:00,mp_1,False,0.955806,NaN
3,2026-04-10 21:46:54.501061+02:00,mp_2,False,NaN,0.955806
4,2026-04-10 22:01:54.501061+02:00,mp_1,False,1.194439,NaN
...,...,...,...,...,...
351,2026-04-14 21:00:00+02:00,mp_2,True,NaN,0.804801
352,2026-04-14 22:00:00+02:00,mp_1,True,0.784051,NaN
353,2026-04-14 22:00:00+02:00,mp_2,True,NaN,0.784051
354,2026-04-14 23:00:00+02:00,mp_1,True,0.886671,NaN


In [ ]:
# consumption, generation
# mp_1_consumption, mp_2_generation, mp_3_generation

## Json

In [ ]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "generate_series",
      "description": "Generate synthetic electricity demand (consumption) time series data.",
      "parameters": {
        "type": "object",
        "properties": {
          "start": {
            "type": "string",
            "format": "date-time",
            "description": "Start timestamp in ISO 8601 format (e.g. 2026-01-01T00:00:00)"
          },
          "end": {
            "type": "string",
            "format": "date-time",
            "description": "End timestamp in ISO 8601 format (e.g. 2026-01-01T23:00:00)"
          },
          "pattern": {
            "type": "string",
            "enum": ["residential", "industrial"],
            "description": "Consumption pattern type. Residential has peaks in the morning and evening, industrial peaks during working hours.",
            "default": "residential"
          }
        },
        "required": ["start", "end"]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "generate_production_series",
      "description": "Generate synthetic electricity supply (production) time series data.",
      "parameters": {
        "type": "object",
        "properties": {
          "start": {
            "type": "string",
            "format": "date-time",
            "description": "Start timestamp in ISO 8601 format (e.g. 2026-01-01T00:00:00)"
          },
          "end": {
            "type": "string",
            "format": "date-time",
            "description": "End timestamp in ISO 8601 format (e.g. 2026-01-01T23:00:00)"
          },
          "pattern": {
            "type": "string",
            "enum": ["residential", "industrial"],
            "description": "Production pattern type. Residential may simulate small-scale generation (e.g. solar), industrial simulates large-scale continuous production.",
            "default": "residential"
          }
        },
        "required": ["start", "end"]
      }
    }
  }
]

In [ ]:
def get_timeseries(
    meter_id: str,
    metric: str,
    start_time: str,
    end_time: str,
    include_forecast: bool = True
):
    pass